# FinSight AI — Research Notebook 04: PyTorch LSTM Deep Learning Architecture
## Training a Recurrent Neural Network with Dropout and Learning Rate Scheduling

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "backend"))  # run from the ml_research/ folder
import numpy as np
import pandas as pd
import torch
from app.data import market_data as md
from app.ml.evaluation import evaluate_model
from app.ml.feature_engineering import FEATURE_COLUMNS, compute_features, create_target, time_series_split
from app.ml.models import LSTMModel

print(f"PyTorch {torch.__version__} · CUDA available: {torch.cuda.is_available()}")

In [ ]:
H = 1
raw, is_demo = md.history("AAPL", "5y", "1d")
feats = compute_features(raw)
X_all, y = create_target(feats, horizon=H)
X = X_all[FEATURE_COLUMNS]
X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(X, y, gap=H)

lstm = LSTMModel("AAPL", horizon=H, sequence_length=20)
lstm.fit(X_train, y_train, X_val, y_val)
print("Hyperparameters:", lstm.hyperparameters)

# predict_full returns one forecast per row (windows may look back into earlier rows' features)
pred_ret = pd.Series(lstm.predict_full(X), index=X.index).loc[X_test.index].values
base = X_all["Close"].loc[X_test.index].values
metrics = evaluate_model(base * (1 + y_test.values), base * (1 + pred_ret), "test", base)
print("LSTM test metrics:", {k: metrics[k] for k in ("rmse", "mape", "directional_accuracy", "skill_vs_naive")})